# Part 3: NLP and Sequence Modeling Mini Project
## Customer Support Sentiment Classification

**Dataset:** Customer Support Text Classification Dataset  
**Target:** `sentiment_label` — `positive`, `neutral`, `negative`  
**Goal:** Build an NLP pipeline using traditional vectorization and sequence-based deep learning.

In [ ]:
import os, re, string, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, precision_score, recall_score)
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
tf.get_logger().setLevel('ERROR')

os.makedirs('results', exist_ok=True)
print('All libraries loaded!')

---
## Task 1: Dataset Understanding

In [ ]:
df = pd.read_csv('customer_support_text_classification.csv')

print('=== Dataset Overview ===')
print(f'Number of records : {df.shape[0]}')
print(f'Number of columns : {df.shape[1]}')
print(f'Columns           : {list(df.columns)}')
print()
print('First 5 rows:')
df.head()

In [ ]:
print('Target Labels (sentiment_label):')
print(df['sentiment_label'].value_counts())
print()
print(f'Average word count : {df["word_count"].mean():.1f} words')
print(f'Min word count     : {df["word_count"].min()}')
print(f'Max word count     : {df["word_count"].max()}')
print()
print('Sample messages:')
for label in ['positive', 'neutral', 'negative']:
    sample = df[df['sentiment_label'] == label]['customer_message'].iloc[0]
    print(f'  [{label.upper()}] {sample}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

counts = df['sentiment_label'].value_counts()
colors = ['#4CAF50', '#9E9E9E', '#F44336']
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black')
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+5, str(v), ha='center', fontweight='bold')
axes[0].set_title('Sentiment Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

for label, color in zip(['positive', 'neutral', 'negative'], colors):
    axes[1].hist(df[df['sentiment_label']==label]['word_count'], alpha=0.6, label=label, color=color, bins=20)
axes[1].set_title('Word Count by Sentiment', fontweight='bold')
axes[1].set_xlabel('Word Count'); axes[1].legend()

channel_counts = df['channel'].value_counts()
axes[2].bar(channel_counts.index, channel_counts.values, color='#1565C0', edgecolor='black')
axes[2].set_title('Messages by Channel', fontweight='bold')

plt.suptitle('Customer Support NLP Dataset — EDA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/eda_plots.png')

**Observations:**
- 1500 records across 3 near-balanced classes (positive: 479, neutral: 524, negative: 497).
- Messages are short — average ~12.7 words, max ~30 words.
- Messages come from 4 channels: chat, phone, email, social.
- The dataset is well-balanced — no class weighting needed.

---
## Task 2: Text Preprocessing

In [ ]:
# Custom stopwords (no NLTK dependency)
STOPWORDS = {
    'i','me','my','myself','we','our','ours','you','your','yours',
    'he','him','his','she','her','hers','it','its','they','them','their',
    'what','which','who','this','that','these','those','am','is','are','was',
    'were','be','been','being','have','has','had','do','does','did','will',
    'would','could','should','may','might','can','a','an','the','and','but',
    'if','or','as','at','by','for','in','of','on','to','up','with','about',
    'into','through','then','once','here','there','when','where','how',
    'all','any','both','each','more','most','other','some','no','not',
    'only','so','than','too','very','s','t','just','now','d','ll','m'
}

def clean_text(text):
    text = str(text).lower()                                     # Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)                  # Remove URLs
    text = re.sub(r'\d+', '', text)                             # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    tokens = text.split()                                        # Tokenize
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]  # Remove stopwords
    return ' '.join(tokens)

df['cleaned_text'] = df['customer_message'].apply(clean_text)

print('=== Preprocessing Examples ===')
for _, row in df.head(4).iterrows():
    print(f'Original : {row["customer_message"]}')
    print(f'Cleaned  : {row["cleaned_text"]}')
    print()

**Preprocessing Steps Applied:**
- **Lowercasing** — standardizes case (`Payment` → `payment`).
- **Remove URLs & numbers** — not informative for sentiment.
- **Remove punctuation** — reduces noise.
- **Tokenization** — splits text into individual words.
- **Stopword removal** — removes common words (the, is, and) that don't carry sentiment meaning.

---
## Task 3: Text Vectorization

In [ ]:
# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['sentiment_label'])
print('Label encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

# ── Approach 1: TF-IDF (uni+bigrams) ──────────────────────────────────────────
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# ── Approach 2: Bag of Words ───────────────────────────────────────────────────
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow  = bow.transform(X_test)

# ── Approach 3: Tokenizer sequences (for LSTM) ────────────────────────────────
MAX_WORDS = 5000
MAX_LEN   = 50
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN, padding='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LEN, padding='post')

print(f'\nTF-IDF matrix  : {X_train_tfidf.shape}')
print(f'BoW matrix     : {X_train_bow.shape}')
print(f'LSTM sequences : {X_train_seq.shape}')

**Why must text be converted to vectors?**

Machine learning models operate on numbers, not raw text. Words like "refund", "payment", and "angry" need to be expressed as numerical arrays before a model can process them. Vectorization bridges the gap between human language and mathematical computation.

| Method | How it works | Best for |
|---|---|---|
| **Bag of Words (BoW)** | Counts how many times each word appears — ignores order | Fast baseline, simple classification |
| **TF-IDF** | Weights words by frequency (TF) and rarity across documents (IDF) — discounts common words | Text classification, search |
| **Tokenizer + Sequences** | Converts words to integer indices preserving order — fed into an Embedding layer | RNN/LSTM models that need word order |

BoW and TF-IDF lose word order. Sequence-based approaches (Tokenizer → LSTM) preserve the sequential nature of language, which matters for sentiment (e.g. "not good" vs "good").

---
## Task 4: Baseline Models

In [ ]:
# Model 1: Logistic Regression + TF-IDF
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)
lr_acc  = accuracy_score(y_test, lr_pred)

print(f'=== Logistic Regression + TF-IDF ===')
print(f'Accuracy: {lr_acc:.4f}')
print(classification_report(y_test, lr_pred, target_names=le.classes_))

In [ ]:
# Model 2: Naive Bayes + Bag of Words
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
nb_pred = nb.predict(X_test_bow)
nb_acc  = accuracy_score(y_test, nb_pred)

print(f'=== Naive Bayes + Bag of Words ===')
print(f'Accuracy: {nb_acc:.4f}')
print(classification_report(y_test, nb_pred, target_names=le.classes_))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pred, title in [(axes[0], lr_pred, 'Logistic Regression + TF-IDF'),
                         (axes[1], nb_pred, 'Naive Bayes + BoW')]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_)
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(y_test, pred):.3f}', fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Baseline Model Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/baseline_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluation table
eval_df = pd.DataFrame([
    {'Model': 'Logistic Regression + TF-IDF',
     'Accuracy': round(lr_acc, 4),
     'Precision': round(precision_score(y_test, lr_pred, average='weighted'), 4),
     'Recall': round(recall_score(y_test, lr_pred, average='weighted'), 4),
     'F1-Score': round(f1_score(y_test, lr_pred, average='weighted'), 4)},
    {'Model': 'Naive Bayes + BoW',
     'Accuracy': round(nb_acc, 4),
     'Precision': round(precision_score(y_test, nb_pred, average='weighted'), 4),
     'Recall': round(recall_score(y_test, nb_pred, average='weighted'), 4),
     'F1-Score': round(f1_score(y_test, nb_pred, average='weighted'), 4)},
])
eval_df.to_csv('results/model_evaluation.csv', index=False)
print('Baseline Model Comparison:')
eval_df

**Interpretation:** Both baseline models achieve very high accuracy on this dataset. This is expected because the customer messages contain highly distinctive vocabulary per sentiment class — positive messages use words like "appreciate", "great", "helpful"; negative messages use "angry", "frustrated", "unacceptable". TF-IDF with Logistic Regression exploits these lexical patterns very effectively, making it a strong baseline for short-text sentiment classification.

---
## Task 5: Sequence Model — LSTM

In [ ]:
y_train_cat = to_categorical(y_train, 3)
y_test_cat  = to_categorical(y_test,  3)

lstm_model = Sequential([
    # Input sequence: integer token IDs, shape = (MAX_LEN,)
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN, name='embedding'),
    # Recurrent layer: processes tokens sequentially, retaining context
    LSTM(64, return_sequences=False, name='lstm'),
    Dropout(0.3, name='dropout'),
    # Dense hidden layer
    Dense(32, activation='relu', name='dense'),
    # Output: 3-class softmax
    Dense(3, activation='softmax', name='output')
], name='SentimentLSTM')

lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',   # Multi-class loss
    metrics=['accuracy']
)
lstm_model.summary()

**LSTM Architecture Explanation:**

| Component | Description |
|---|---|
| **Input sequence** | Each message → list of integer token IDs, padded to length 50 |
| **Embedding layer** | Converts each integer ID into a 64-dim dense vector. These vectors are learned during training — similar words end up close together |
| **LSTM layer** | Processes the sequence of embeddings one token at a time, maintaining a hidden state (memory) across the sequence |
| **Dropout** | Regularization — randomly zeros 30% of connections to prevent overfitting |
| **Dense output** | Combines LSTM output → 3 logits → Softmax probabilities |
| **Loss function** | Categorical Cross-Entropy — standard for multi-class classification |
| **Evaluation metric** | Accuracy + F1-Score (per-class) |


In [ ]:
history_lstm = lstm_model.fit(
    X_train_seq, y_train_cat,
    epochs=15, batch_size=32,
    validation_split=0.1, verbose=1
)

lstm_loss, lstm_acc = lstm_model.evaluate(X_test_seq, y_test_cat, verbose=0)
lstm_pred = np.argmax(lstm_model.predict(X_test_seq, verbose=0), axis=1)
print(f'\nLSTM Test Accuracy: {lstm_acc:.4f}')
print(classification_report(y_test, lstm_pred, target_names=le.classes_))

In [ ]:
# LSTM training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(history_lstm.history['loss'],     label='Train Loss', color='#1565C0')
axes[0].plot(history_lstm.history['val_loss'], label='Val Loss',   color='#F44336', linestyle='--')
axes[0].set_title('LSTM Loss over Epochs', fontweight='bold'); axes[0].legend()

axes[1].plot(history_lstm.history['accuracy'],     label='Train Acc', color='#1565C0')
axes[1].plot(history_lstm.history['val_accuracy'], label='Val Acc',   color='#F44336', linestyle='--')
axes[1].set_title('LSTM Accuracy over Epochs', fontweight='bold'); axes[1].legend()

plt.suptitle('LSTM Training Curves — Sentiment Classification', fontweight='bold')
plt.tight_layout()
plt.savefig('results/lstm_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Full model comparison
eval_df_full = pd.concat([eval_df, pd.DataFrame([{
    'Model': 'LSTM (Embedding + Sequence)',
    'Accuracy': round(lstm_acc, 4),
    'Precision': round(precision_score(y_test, lstm_pred, average='weighted'), 4),
    'Recall': round(recall_score(y_test, lstm_pred, average='weighted'), 4),
    'F1-Score': round(f1_score(y_test, lstm_pred, average='weighted'), 4)
}])], ignore_index=True)
eval_df_full.to_csv('results/model_evaluation.csv', index=False)
print('Full Model Comparison:')
eval_df_full

**Why the LSTM underperforms here:**

The LSTM struggles (~35% accuracy — near random) while the baseline models excel. This is because:
- Messages are very short (avg 12.7 words) — sequence modeling has little temporal structure to exploit.
- The dataset has very distinctive vocabulary per sentiment class, which TF-IDF captures perfectly without needing word order.
- The LSTM would need more training data and longer sequences to generalize. In real-world NLP, pre-trained embeddings (GloVe, Word2Vec) or transformers (BERT) would be used instead.

**Key insight:** Traditional models (LR + TF-IDF) are often competitive or superior for short-text classification. LSTMs shine in longer texts where word order and long-range dependencies matter.

---
## Task 6: Attention and Transformer Reflection

### Why do RNNs struggle with long-term dependencies?

A vanilla RNN processes text sequentially — it passes a hidden state from one word to the next. In a long sentence, information from early words must travel through many intermediate steps to reach the final state. During backpropagation, gradients are multiplied at each step; with many steps, they either shrink to nearly zero (**vanishing gradients**) or explode. This makes it nearly impossible for RNNs to learn that a word at position 1 influences the meaning at position 50.

### How do LSTMs help with memory?

LSTMs solve the vanishing gradient problem with a **cell state** — a dedicated memory lane that runs alongside the hidden state. Three learnable **gates** control what information is retained or discarded:
- **Forget gate:** Decides what to erase from memory.
- **Input gate:** Decides what new information to write to memory.
- **Output gate:** Decides what to read from memory for the next prediction.

Because the cell state is only modified additively (not multiplicatively at every step), gradients flow much more easily over long sequences, enabling the model to remember relevant context from earlier in the text.

### What does Attention solve in sequence-to-sequence tasks?

In classic encoder-decoder RNNs (e.g. for machine translation), the entire input sentence is compressed into a single fixed-size vector. For long sentences, this bottleneck loses important details. **Attention** allows the decoder to look back at all encoder hidden states and compute a weighted combination — paying more attention to the relevant input words for each output word. This dramatically improves translation quality and removes the information bottleneck.

For example, when generating the French word "chercher" for the English "look for", attention lets the decoder directly focus on those two input words rather than relying on a single compressed summary.

### Why are Transformers important in modern NLP and Generative AI?

Transformers replaced RNNs and LSTMs as the dominant NLP architecture. Key reasons:

| Advantage | Explanation |
|---|---|
| **Parallelism** | RNNs process tokens sequentially — can't be parallelized. Transformers process all tokens simultaneously, making training on large datasets practical. |
| **Self-attention** | Every word attends to every other word directly — no vanishing gradient over long distances. |
| **Scalability** | Transformers scale remarkably well — larger models + more data = dramatically better performance (GPT-4, Claude, Gemini). |
| **Pre-training** | BERT, GPT, and others are pre-trained on massive corpora and fine-tuned for specific tasks, enabling transfer learning in NLP. |
| **Generative AI** | The decoder-only transformer (GPT architecture) enables autoregressive text generation — the foundation of ChatGPT, Claude, Gemini, and all modern large language models. |

The transformer's **"Attention is All You Need"** (Vaswani et al., 2017) is arguably the most influential paper in modern AI, enabling everything from BERT for classification to GPT-4 for generation.

In [ ]:
# Save sample predictions
sample_msgs   = df['customer_message'].iloc[:10].tolist()
sample_cleaned = df['cleaned_text'].iloc[:10].tolist()
sample_true   = le.inverse_transform(df['label'].iloc[:10])
sample_tfidf  = tfidf.transform(sample_cleaned)
sample_pred   = le.inverse_transform(lr.predict(sample_tfidf))

with open('results/sample_predictions.txt', 'w') as f:
    f.write('SAMPLE PREDICTIONS — Logistic Regression + TF-IDF\n')
    f.write('=' * 60 + '\n\n')
    for i, (msg, true, pred) in enumerate(zip(sample_msgs, sample_true, sample_pred)):
        status = 'CORRECT' if true == pred else 'WRONG'
        f.write(f'[{i+1}] {status}\n')
        f.write(f'  Message   : {msg}\n')
        f.write(f'  True Label: {true}\n')
        f.write(f'  Predicted : {pred}\n\n')

print('Sample predictions saved to results/sample_predictions.txt')